In [2]:
import pandas as pd
import numpy as np

arquivo_limpo = "../dados/raw/rais_sp_limpo.csv"

# Define tipos compactos para carregar 14,9 milhões de linhas usando pouca memória RAM
tipos = {
    "genero": "category",
    "salario_mensal": "float32",
    "cbo_ocupacao": "int32",
    "horas_semanais": "float32",
    "escolaridade_cod": "int8",
}

print("Carregando base limpa...")
df = pd.read_csv(arquivo_limpo, dtype=tipos)

# Remove possíveis outliers extremos de erro de digitação (ex: salários acima de R$ 150 mil/mês)
# Bom ponto para citar: tratamento de outliers para evitar distorção na média
df = df[(df["salario_mensal"] >= 500) & (df["salario_mensal"] <= 150_000)].copy()

# Calcula o salário-hora estimado (mês comercial padrão = 4,33 semanas)
df["salario_hora"] = np.where(
    df["horas_semanais"] > 0,
    df["salario_mensal"] / (df["horas_semanais"] * 4.33),
    np.nan,
)

# 1. Estatísticas Gerais por Gênero
resumo = (
    df.groupby("genero", observed=True)["salario_mensal"]
    .agg(
        total_vinculos="count",
        media="mean",
        desvio_padrao="std",
        q1_25=lambda s: np.percentile(s, 25),
        mediana_q2="median",
        q3_75=lambda s: np.percentile(s, 75),
    )
    .reset_index()
)

resumo["participacao_pct"] = (
    resumo["total_vinculos"] / resumo["total_vinculos"].sum() * 100
).round(2)

# Adiciona mediana do salário-hora
mediana_hora = (
    df.groupby("genero", observed=True)["salario_hora"]
    .median()
    .rename("mediana_salario_hora")
)
resumo = resumo.merge(mediana_hora, on="genero")

print("\n=== RESUMO ESTATÍSTICO SALARIAL (SP) ===")
print(resumo.to_string(index=False))

# 2. Cálculo do Gender Pay Gap (%)
med_m = resumo.loc[resumo["genero"] == "Masculino", "mediana_q2"].values[0]
med_f = resumo.loc[resumo["genero"] == "Feminino", "mediana_q2"].values[0]

media_m = resumo.loc[resumo["genero"] == "Masculino", "media"].values[0]
media_f = resumo.loc[resumo["genero"] == "Feminino", "media"].values[0]

gap_mediana = ((med_m - med_f) / med_m) * 100
gap_media = ((media_m - media_f) / media_m) * 100

print(f"\nGap Salarial (Mediana): Mulheres ganham {gap_mediana:.2f}% a menos que homens.")
print(f"Gap Salarial (Média):   Mulheres ganham {gap_media:.2f}% a menos que homens.")

# 3. Gap Salarial por Nível de Escolaridade (Agrupado)
mapa_escolaridade = {
    1: "1. Fundamental Incompleto ou Menos",
    2: "1. Fundamental Incompleto ou Menos",
    3: "1. Fundamental Incompleto ou Menos",
    4: "1. Fundamental Incompleto ou Menos",
    5: "2. Fundamental Completo",
    6: "3. Médio Incompleto",
    7: "4. Médio Completo",
    8: "5. Superior Incompleto",
    9: "6. Superior Completo",
    10: "7. Pós-Graduação (Mestrado/Doutorado)",
    11: "7. Pós-Graduação (Mestrado/Doutorado)",
}

df["escolaridade_grupo"] = df["escolaridade_cod"].map(mapa_escolaridade)

por_escolaridade = (
    df.dropna(subset=["escolaridade_grupo"])
    .groupby(["escolaridade_grupo", "genero"], observed=True)["salario_mensal"]
    .median()
    .unstack()
    .reset_index()
)

por_escolaridade["gap_mediana_pct"] = (
    (por_escolaridade["Masculino"] - por_escolaridade["Feminino"])
    / por_escolaridade["Masculino"]
    * 100
).round(2)

print("\n=== MEDIANA SALARIAL E GAP POR ESCOLARIDADE ===")
print(por_escolaridade.to_string(index=False))

# Salva tabelas resumidas leves (prontas para dashboards e cruzamento com a C&A)
resumo.to_csv("resumo_geral_salarios.csv", index=False)
por_escolaridade.to_csv("resumo_salarios_escolaridade.csv", index=False)
print("\nArquivos 'resumo_geral_salarios.csv' e 'resumo_salarios_escolaridade.csv' salvos com sucesso!")

Carregando base limpa...

=== RESUMO ESTATÍSTICO SALARIAL (SP) ===
   genero  total_vinculos       media  desvio_padrao       q1_25  mediana_q2       q3_75  participacao_pct  mediana_salario_hora
 Feminino         6919507 4356.864258    5278.237305 2016.719971 2655.439941 4708.390137             46.33             14.592104
Masculino         8014976 5193.161133    6755.969238 2295.860107 3210.030029 5305.870117             53.67             17.077736

Gap Salarial (Mediana): Mulheres ganham 17.28% a menos que homens.
Gap Salarial (Média):   Mulheres ganham 16.10% a menos que homens.

=== MEDIANA SALARIAL E GAP POR ESCOLARIDADE ===
                   escolaridade_grupo     Feminino    Masculino  gap_mediana_pct
   1. Fundamental Incompleto ou Menos  2068.765137  2940.000000        29.629999
              2. Fundamental Completo  2127.790039  2901.840088        26.670000
                  3. Médio Incompleto  1964.170044  2469.979980        20.480000
                    4. Médio Completo 